In [ ]:
# use a conda torch gpu environment
# extra deps:
%pip install sentence-transformers chromadb

Pre-Process text

In [3]:
import csv
import os

if not os.path.exists("test_notebooks"):
    os.chdir("..")

assert os.path.exists("test_notebooks")

In [22]:
from pathlib import Path
from typing import List


lines = Path("datasets/danbooru-10w.txt").read_text("utf8").splitlines()
reader = csv.reader(lines)

rows:List[dict] = []

for i, item in enumerate(reader):
    original = item[0]
    count = item[1]
    space_tag = original.replace("_", " ")

    rows.append({
        "id": str(i),
        "original": original,
        "count": count,
        "space_tag": space_tag,
    })

rows[:5]

[{'id': '0', 'original': '1girl', 'count': '4114588', 'space_tag': '1girl'},
 {'id': '1', 'original': 'solo', 'count': '3426446', 'space_tag': 'solo'},
 {'id': '2',
  'original': 'highres',
  'count': '3008413',
  'space_tag': 'highres'},
 {'id': '3',
  'original': 'long_hair',
  'count': '2898315',
  'space_tag': 'long hair'},
 {'id': '4',
  'original': 'commentary_request',
  'count': '2610959',
  'space_tag': 'commentary request'}]

In [5]:
import torch
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions
import numpy as np
from tqdm import tqdm

c:\Users\ThePlayer\miniconda3\envs\comfy\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [7]:
model = SentenceTransformer('BAAI/bge-m3', device=device)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 744.12it/s, Materializing param=pooler.dense.weight]                               


In [8]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 8192, 'do_lower_case': False, 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [9]:
client = chromadb.PersistentClient(path="./database/chroma_1")

In [42]:
client.delete_collection(name="danbooru_tags")

collection = client.get_or_create_collection(name="danbooru_tags",metadata={"hnsw:space": "cosine"})

In [ ]:
batch_size = 512 

subset = rows[:]

for i in tqdm(range(0, len(subset), batch_size)):
    batch = subset[i : i + batch_size]
    batch_tags = [p['original'] for p in batch]
    batch_processed = [p['space_tag'] for p in batch]
    ids = [str(p['id']) for p in batch]
    
    # 生成向量 (BGE-M3 默认输出 1024 维)
    # normalize_embeddings=True 对余弦相似度检索非常重要
    with torch.no_grad():
        embeddings = model.encode(
            batch_processed, 
            batch_size=batch_size, 
            normalize_embeddings=True
        ).tolist()

    
    # 插入 ChromaDB
    collection.add(
        embeddings=embeddings,
        documents=batch_tags, # 原始带下划线的标签
        ids=ids
    )

print(f"成功导入 {collection.count()} 个标签")

  9%|▊         | 17/196 [00:43<09:40,  3.24s/it]

In [41]:
query_text = "乐器工具"
query_embedding = model.encode(query_text, normalize_embeddings=True).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=20
)

print("检索到的标签:", results['documents'])

检索到的标签: [['instrument', 'playing_instrument', 'holding_instrument', 'guitar', 'music', 'earphones', 'drumsticks', 'violin', 'headphones', 'piano', 'audible_music', 'microphone', 'sound_effects', 'electric_guitar', 'weapon', 'musical_note', 'bass_guitar', 'jewelry', 'spatula', 'earbuds']]


In [ ]:
# 